<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/06-harness-engineering.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Harness 工程

**目标：** 给这一整节一直在逐个零件教你的这门手艺起个名字（对模型调用*外围脚手架*的工程化），并补上它还没有明说的三个杠杆：上下文装配、工具结果整形、验证循环。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 第三条轴

现在你已经有三种改变 LLM 系统行为的方式，而且它们彼此独立：

- **Prompt 工程** —— 你发送的*消息*。（第 01 节，贯穿全篇。）
- **模型适配** —— 调用背后的*权重*。（第 06 节：prompt → RAG → 微调。）
- **Harness 工程** —— 调用*外围的脚手架*：你每轮如何装配上下文、如何整形工具返回的内容、循环及其停止条件，以及验证模型成果的检查。

一个已上线 agent 的可靠性，大部分来自第三种，而它正是职位名称很少点出来的那一块。关键在于：**第 05 节一直在教 harness 工程**，只是一个组件一个组件地教。

| 你搭过的东西 | Harness 组件 |
|---|---|
| [01 Agent 循环](01-agent-loop-from-scratch.zh.ipynb) | 循环，模型运行其间的骨架 |
| [02 工具设计](02-tool-design.zh.ipynb) | 工具的*输入*侧：模型能调用什么 |
| [03 护栏与预算](03-guardrails-and-budgets.zh.ipynb) | 停止条件、边界、循环检测 |
| [04 MCP](04-mcp-and-the-tool-ecosystem.zh.ipynb) | 把*外部*工具接入 harness |
| [05 Skills](05-skills-and-progressive-disclosure.zh.ipynb) | 打包 harness 按需加载的知识 |

本 notebook 给这门手艺命名，并补上这一节留而未明的三个空缺：

1. **上下文装配** —— *每轮*到底有什么进入窗口（以及当它变大时该丢掉什么）。
2. **工具结果整形** —— 工具运行之后你*回填*什么（工具设计的返回侧）。
3. **验证循环** —— harness 检查模型的成果，并在失败时循环，而不只是在工具调用上循环。

> **⭐ 关键结论 ——** 模型是固定的；harness 是你的。当 agent 行为不端时，先去动脚手架（上下文、工具结果、循环、检查），再去考虑换更大的模型或做微调。

## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端，这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 同样的工具，同样的循环

我们复用第 01 节的假文件系统、它的四个工具，以及一个紧凑版的循环，这样下面每一处改动都明显地是对*harness*的改动，而不是对任务的改动。这里没有新东西；扫一眼就往下走。

In [ ]:
import json, re

# 第 01 节的假文件系统：path -> content。安全、可检视、可重置。
FS = {
    "notes/monday.md":    "Team lunch: $42.50\nTaxi to client site: $18.00\n",
    "notes/tuesday.md":   "Conference tickets, 2 x $150.00 each\n",
    "notes/wednesday.md": "Cloud credits top-up: $75.25\nCoffee for the workshop: $23.10\n",
}

def list_files():
    return "\n".join(sorted(FS)) if FS else "(no files)"

def read_file(path):
    if path not in FS:
        return f"Error: no file at '{path}'. Call list_files to see what exists."
    return FS[path]

def write_file(path, content):
    FS[path] = content
    return f"Wrote {len(content)} characters to {path}."

def calculator(expression):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Error: only numbers and + - * / ( ) are allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

IMPLS = {"list_files": list_files, "read_file": read_file,
         "write_file": write_file, "calculator": calculator}

def execute_tool(name, args):
    fn = IMPLS.get(name)
    if fn is None:
        return f"Error: unknown tool '{name}'."
    try:
        return str(fn(**args))
    except TypeError as e:
        return f"Error: bad arguments for {name}: {e}"

def _tool(name, desc, props, required):
    return {"type": "function", "function": {
        "name": name, "description": desc,
        "parameters": {"type": "object", "properties": props, "required": required}}}

TOOLS = [
    _tool("list_files", "List every file path in the workspace, one per line.", {}, []),
    _tool("read_file", "Read the full contents of one file.",
          {"path": {"type": "string"}}, ["path"]),
    _tool("write_file", "Create or overwrite a file with the given content.",
          {"path": {"type": "string"}, "content": {"type": "string"}}, ["path", "content"]),
    _tool("calculator", "Evaluate an arithmetic expression with + - * / and parentheses.",
          {"expression": {"type": "string"}}, ["expression"]),
]

def run(messages, tools=TOOLS, shape_result=lambda name, out: out, max_turns=8):
    """第 01 节的循环，紧凑版，只多了一个新接缝：shape_result 让
    harness 在每个工具的输出重新进入上下文之前对它做变换。
    """
    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages)
        choice = resp.choices[0]
        msg = choice.message
        if choice.finish_reason != "tool_calls":
            return msg.content or "", messages
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            out = shape_result(tc.function.name, execute_tool(tc.function.name, args))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})
    return "[hit max_turns]", messages

print("scaffold ready:", list(IMPLS))

## 杠杆 1 —— 上下文装配：*每轮*进入窗口的是什么

API 是无状态的。每一轮都会重发**整个** `messages` 列表；这正是 notebook 01 和 03 反复点出的观察（「逐轮成本上升，因为你在重发整段历史」）。所以上下文窗口不是模型*拥有*的一个地方；它是你的 harness *在每一次调用时重建并为之付费*的东西。

这个重新框定交给你一个杠杆。由你决定什么能活到下一轮。在一次长运行里（几十次工具调用、大文件读取），原始记录会冲出窗口，账单一轮比一轮高。harness 的动作是**压缩**：一旦记录越过一个阈值，就用一段短摘要替换掉旧的中段，只保留头部（任务）和尾部（最近的轮次）。

下面这个单元格度量一份记录，然后压缩它，不需要任何模型调用就能看清要点。

In [ ]:
def transcript_size(messages):
    """粗略的 token 代理：字符数/4。足以看出趋势。"""
    chars = sum(len(json.dumps(m, default=str)) for m in messages)
    return len(messages), chars, chars // 4

def compact(messages, keep_recent=4, summary=None):
    """保留第一条消息（任务）+ 最后 `keep_recent` 条；把中段替换成
    一行摘要。真实的 harness 会请模型来写这段摘要；这里我们合成它，
    好让机制可见且免费。
    """
    if len(messages) <= keep_recent + 1:
        return messages
    dropped = messages[1:-keep_recent]
    note = summary or f"[{len(dropped)} earlier turns compacted: files read and totals computed so far]"
    return [messages[0], {"role": "user", "content": note}, *messages[-keep_recent:]]

# 模拟一次长运行：先是任务，然后是很多体积庞大的工具往来。
long_run = [{"role": "user", "content": "Audit every notes file and reconcile the totals."}]
for i in range(20):
    long_run.append({"role": "assistant", "content": None,
                     "tool_calls": [{"id": f"c{i}", "type": "function",
                        "function": {"name": "read_file", "arguments": '{"path":"notes/x.md"}'}}]})
    long_run.append({"role": "tool", "tool_call_id": f"c{i}",
                     "content": "Some line item: $" + "1234.56, " * 20})  # 体积庞大的结果

before = transcript_size(long_run)
after  = transcript_size(compact(long_run))
print(f"before compaction: {before[0]:>3} msgs, ~{before[2]:>5} tokens")
print(f"after  compaction: {after[0]:>3} msgs, ~{after[2]:>5} tokens")
print(f"→ {100 * (before[2]-after[2]) // before[2]}% smaller, task + recent turns preserved")

> **⚠️ 生产现实 ——** 压缩是刻意有损的，而*丢掉什么*是一个设计决策。把中段总结错了，agent 就会忘记它在 15 轮前答应过的一个约束。保留任务、保留最近的轮次、保留模型承诺过的任何东西（一个计划、一个滚动累计值）；其余的做摘要。这和 [05 Skills](05-skills-and-progressive-disclosure.zh.ipynb) 里「上下文是一份预算」的想法一样，只是用在对话记录上，而不是加载进来的知识上。

## 杠杆 2 —— 工具结果整形：你*回填*的是什么

notebook 02 设计的是工具的**输入**侧：名字、描述、模型调用前读的 schema。这里是**输出**侧：你的 harness 在结果*变成下一轮上下文之前*对它做了什么。模型从来看不到原始返回值；它看到的是你作为 `tool` 消息追加进去的任何东西。有两个整形动作值得留下：

- **截断大块数据。** 一个转储 50 KB 文件的工具会毒害之后每一轮（按杠杆 1，你每次都要重发它）。头部 + 尾部 + 一个诚实的「省略 N 个字符」标记，既保住信号又封住成本。
- **让错误可操作。** 一个模型能*据以行动*的错误（「no file at 'x'; call list_files」）把一次死掉的轮次变成一次恢复。一段原始堆栈跟踪或一个光秃秃的 `None` 只会烧掉轮次。第 01 节的工具已经这么做了；整形是你集中强制这一点的地方。

上面的 `run()` 循环已经有这个接缝，也就是 `shape_result` 钩子。下面我们传入一个，看一个巨大的结果被截断。

In [ ]:
# 一个太大、不适宜每轮重发的工具结果。
FS["notes/dump.md"] = "line item $9.99\n" * 400  # 约 6 KB

def shape(name, out, limit=300):
    """在每一个工具结果重新进入上下文之前，对它施加的 harness 级整形。"""
    if len(out) <= limit:
        return out
    head, tail = out[:limit // 2], out[-limit // 2:]
    return f"{head}\n...[{len(out) - limit} chars omitted by the harness]...\n{tail}"

raw = read_file("notes/dump.md")
shaped = shape("read_file", raw)
print(f"raw result:    {len(raw):>5} chars  (re-sent every turn if unshaped)")
print(f"shaped result: {len(shaped):>5} chars\n")
print(shaped[:220], "...")

# 把整形接上去跑真正的 agent —— 那个巨大的文件不再撑爆上下文。
FS.pop("summary.md", None)
answer, _ = run(
    [{"role": "user", "content": "Read notes/dump.md and tell me roughly how many line items it has."}],
    shape_result=shape)
print("\n=== answer ===\n", answer)

## 杠杆 3 —— 验证循环：harness 来检查成果

notebook 01 的循环只在一个条件上停止：*模型没有请求工具。* 但「模型宣布自己做完了」并不等于「成果是对的」。harness 可以设一条更高的门槛：对结果运行一个**检查**，失败时把具体问题作为新的一轮回填给模型，让它去修。这就是 [03 的轨迹评测](03-guardrails-and-budgets.zh.ipynb) 的想法指向*输出*：一个由 harness 拥有的断言，而不是寄希望于模型做对了。

任务是：把开销加起来并写出 `summary.md`。harness 强制的检查是：**`summary.md` 里的数字必须实际等于各项开销之和。** 如果不是，harness 精确说明哪里不对并循环（当然是有界的；来自 03 的杠杆：每个循环都需要一个上限）。

In [ ]:
def truth_total():
    """harness 自己的地面真值 —— 用代码算出来，不采信模型的说法。"""
    amounts = [float(x) for f, c in FS.items() if f.startswith("notes/") and f != "notes/dump.md"
               for x in re.findall(r"\$?([0-9]+\.[0-9]{2})", c)]
    # tuesday.md 写的是 "2 x $150.00" —— 正则只看到一个 150.00，所以把第二个加上。
    return round(sum(amounts) + 150.00, 2)

def written_total():
    """取出模型写进 summary.md 的那个数字，没有则返回 None。"""
    m = re.findall(r"([0-9]+\.[0-9]{2})", FS.get("summary.md", ""))
    return float(m[-1]) if m else None

def run_verified(task, max_repairs=2):
    messages = [{"role": "user", "content": task}]
    for attempt in range(max_repairs + 1):
        answer, messages = run(messages, shape_result=shape)
        want, got = truth_total(), written_total()
        if got is not None and abs(got - want) < 0.01:
            return f"verified after {attempt} repair(s): total {got} is correct"
        # 验证失败 —— 把那个具体差距作为下一轮回填。
        messages.append({"role": "user", "content":
            f"That's not right. The verified total is {want:.2f} but summary.md "
            f"{'has ' + format(got, '.2f') if got is not None else 'has no total'}. "
            "Recompute with the calculator and rewrite summary.md."})
    return f"still wrong after {max_repairs} repairs (wanted {want}, got {written_total()})"

FS.pop("summary.md", None)
print(run_verified(
    "Read every notes/*.md file (ignore notes/dump.md), add up all expenses with the "
    "calculator, and write summary.md stating the total. Note: 'notes/tuesday.md' lists "
    "'2 x $150.00' — that's two items."))
print("summary.md:", FS.get("summary.md", "(not written)").strip())

> **🚩 常见错误 ——** 相信模型自己说的「做完了」。模型说*「我已经把总额写进 summary.md 了」*是一个声明，不是保证；它可能把算术打错了，或者写错了数字。一个用代码算出地面真值、循环到对得上为止的 harness，就是 demo 和「一件你敢让它无人值守运行的东西」之间的差别。能用代码验证的就用代码验证；把 [04 LLM 作为评判模型](../04-evals/02-llm-as-judge.zh.ipynb) 的方法留给验证不了的。

## harness 才是你要工程化的那个单元

退一步看，这一节读起来是一个整体。agent 不是「一个聪明的模型」；它是**一个模型加一个 harness**，而每一项可靠性都住在 harness 里：

```
task ─▶ ┌────────── HARNESS — your code (the model is fixed) ──────────┐ ─▶ result
        │                                                              │
        │  (1) assemble        ┌────────────┐        (2) shape         │
        │      context ──────▶ │ MODEL call │ ──────▶ tool results ─┐  │
        │         ▲            └────────────┘                       │  │
        │         │                                          (3) verify│
        │         └───── loop: bounded + guarded ◀──────────────────┘  │
        │               (nb 01 loop · nb 03 bounds)                    │
        │                                                              │
        └──────────────────────────────────────────────────────────────┘

  (1) context assembly    (2) result shaping    (3) verification loop
```

中间的模型是固定的。其余一切都是你拥有并测试的代码（你放在它前面的东西、你回填的东西、你何时停下、你检查什么）。这就是为什么「harness 工程」值得命名：当系统不稳定时，修复几乎总是 harness 的改动（更紧的上下文、整形过的结果、一个真正的验证步骤、一个更好的边界），*而不是*一个更大的模型。脚手架对了之后，才去动权重（第 06 节）。

## 实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 把上下文窗口当作你要重建并付费的逐轮预算 | 假定模型「记得」，让对话记录无限增长 |
| 压缩长记录：保留任务 + 最近轮次 + 承诺，对中段做摘要 | 盲目截断（丢掉最旧的 N 条），丢失 agent 答应过的约束 |
| 每个工具结果在重新进入上下文之前都整形：限制大小，保留头部+尾部 | 回填原始 50 KB 大块，并每轮重发它们 |
| 返回模型能据以行动的错误（「no such file; call list_files」） | 抛出光秃秃的堆栈跟踪或 `None`，白烧一轮 |
| 用代码验证输出，失败就循环，且有界 | 相信模型自己说的「做完了」；把第一个看着对的答案发出去 |
| 在 harness 里算地面真值，而不是采信模型的声明 | 问模型它自己的答案对不对，然后信它 |
| 不稳定时先修 harness（上下文、结果、检查、边界） | 在脚手架对了之前就去换更大的模型或做微调 |

## 框架什么时候登场：LangChain 与 LangGraph

notebook 01 已经点出 agent 框架在原生循环之上加的东西（持久化、可观测性、交接、运维管路）。现在你连工具设计、护栏、MCP、skills 和 harness 都搭过了，下面这些**简历关键词**如何映射到你搭过的东西，好让关键词和理解一起走：

- **LangChain** —— 组件层：LLM 封装、工具/`@tool` 定义、prompt 模板、记忆。它的 `AgentExecutor` *就是*第 05 节的循环；它的工具抽象*就是*你的工具 schema + 分发（notebook 02）。
- **LangGraph** —— 编排层：你的循环被表达成一张显式的**状态图**（节点 = 步骤，边 = 转移），内建检查点和人在环暂停。当 notebook 06 的 harness（上下文装配、验证循环、有界重试）需要*持久化且可恢复*、而不只是驻留内存时，你就要用它。
- **LlamaIndex** —— 在 RAG 侧有重叠（见第 03 节的桥接）；它的 agent 接的是检索优先的工作流。

> **⭐ 关键结论 ——** 这些框架跑的是你用约 40 行写出的*同一个* while 循环。它们加上持久化、追踪和多 agent 接线，绝不是一种新的魔法。亲手搭过这个循环、它的护栏和它的 harness 之后，你能读懂一张 LangGraph 图就看见你自己的代码，并按精确的维度评判：它持久化什么、给我看什么、隐藏什么？

**什么时候该用：** 需要持久/可恢复的运行、团队需要共享追踪，或者多 agent 交接。**什么时候不该用：** 一个有界、单一用途的 agent，用显式循环往往更清晰也更便宜，并且记得第 03/05 节的纪律：当步骤已知时，*pipeline* 胜过 agent（[护栏](03-guardrails-and-budgets.zh.ipynb)）。

> **🔵 面试信号 ——** 不靠框架的路线从来不是反框架。它让你能说「我用 LangGraph，而且我能告诉你它哪些特性我真的需要、哪些我会跳过」，这正是高级 AI 工程师被雇来做的判断。

## 练习

1. **模型写的摘要。** 把 `compact()` 里合成的字符串换成一次真正的模型调用：当记录越过阈值时，请模型（不带工具、`max_tokens` 设小）用两句话总结被丢掉的中段，再把*那段*接进去。对比省下的 token，以及 agent 是否仍能正确完成任务。
2. **按工具分别整形。** `shape()` 对所有工具一视同仁。把它改成一个逐工具整形器的 dict，例如 `read_file` 用头部+尾部截断，`list_files` 从不截断，`calculator` 原样透传。展示一个任务，其中统一上限有害，而逐工具策略修好了它。
3. **也验证轨迹。** `run_verified` 检查的是*输出*。再加一个对*路径*的检查（来自 [03 的轨迹评测](03-guardrails-and-budgets.zh.ipynb)）：如果 agent 写了 `summary.md` 却从没调用 `calculator`，就判失败并修复。确认两个检查都通过之后，它才返回 `verified`。
4. **故意弄坏它。** 把 `max_repairs=0`，并把 `truth_total()` 改成一个错误的期望值。看 harness 干净、具体地报告失败，而不是悄悄接受坏输出，然后写一句话说明为什么「带着差距大声失败」胜过「把模型最后写的东西原样返回」。